# 01 Data integrity

In [2]:
import glob
import pandas as pd

ROOT = "../data/ContentWiseImpressions/data/ContentWiseImpressions/CW10M"
inter = pd.read_parquet(f"{ROOT}/interactions").reset_index()
direct = pd.read_parquet(f"{ROOT}/impressions-direct-link").reset_index()
nondirect_parts = sorted(glob.glob(f"{ROOT}/impressions-non-direct-link/*.parquet"))

In [3]:
inter.isna().sum()

utc_ts_milliseconds    0
user_id                0
item_id                0
series_id              0
episode_number         0
series_length          0
item_type              0
recommendation_id      0
interaction_type       0
vision_factor          0
explicit_rating        0
dtype: int64

In [4]:
direct.isna().sum()

recommendation_id             0
row_position                  0
recommendation_list_length    0
recommended_series_list       0
dtype: int64

In [5]:
sum(pd.read_parquet(p).reset_index().isna().sum() for p in nondirect_parts)

user_id                       0
row_position                  0
recommendation_list_length    0
recommended_series_list       0
dtype: int64

In [6]:
inter[['item_type', 'interaction_type']].apply(pd.Series.value_counts)

,item_type,interaction_type
0,987518,6122105
1,231290,4105530
2,162574,9109
3,9076428,221066


In [7]:
inter.loc[inter.interaction_type == 0, 'vision_factor'].describe()

count    6.122105e+06
mean     8.302285e-01
std      3.058420e-01
min      1.000000e-02
25%      8.300000e-01
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
Name: vision_factor, dtype: float64

In [8]:
inter.loc[inter.interaction_type != 0, 'vision_factor'].value_counts()

vision_factor
-1.0    4335705
Name: count, dtype: int64

In [9]:
inter.loc[inter.interaction_type == 2, 'explicit_rating'].value_counts().sort_index()

explicit_rating
0.5     205
1.0     113
1.5      66
2.0     155
2.5     136
3.0    3445
3.5     563
4.0    1244
4.5     652
5.0    2530
Name: count, dtype: int64

In [10]:
inter.duplicated().sum()

np.int64(938)

In [11]:
linked = inter.recommendation_id != -1
linked.mean()

np.float64(0.06480209527616203)

In [12]:
inter.loc[linked, 'recommendation_id'].isin(direct.recommendation_id).mean()

np.float64(1.0)

In [13]:
direct.recommendation_id.isin(inter.recommendation_id).mean(), direct.recommendation_id.duplicated().sum()

(np.float64(1.0), np.int64(0))

In [14]:
nondirect_users = pd.concat(pd.read_parquet(p, columns=['row_position']).reset_index().user_id.drop_duplicates() for p in nondirect_parts).unique()
pd.Series(nondirect_users).isin(inter.user_id).mean()

np.float64(1.0)

In [15]:
(direct.recommended_series_list.apply(len) != direct.recommendation_list_length).sum(), (direct.row_position < 0).sum()

(np.int64(0), np.int64(0))

In [16]:
bad = 0
for p in nondirect_parts:
    d = pd.read_parquet(p, columns=['recommended_series_list', 'recommendation_list_length', 'row_position'])
    bad += ((d.recommended_series_list.apply(len) != d.recommendation_list_length) | (d.row_position < 0)).sum()
bad

np.int64(0)

In [17]:
pd.to_datetime(inter.utc_ts_milliseconds, unit='ms').agg(['min', 'max'])

min   2019-01-07 09:00:02
max   2019-04-15 08:59:59
Name: utc_ts_milliseconds, dtype: datetime64[ns]